# 03 - Merge Pre-1997 Legacy Formal Decisions into EC Antitrust Master

## Ziel dieses Notebooks

Dieses Notebook integriert die **Legacy Formal Decisions vor 1997** direkt in die bestehende **EC-Antitrust-Masterdatei**.

**Was dieses Notebook tut:**
- Laedt die bestehende EC-Masterdatei (`ec_antitrust_master.csv`)
- Laedt die Legacy Formal Decisions vor 1997 (`ec_legacy_formal_decisions.csv`)
- Vergleicht ihre Spaltenstruktur transparent
- Baut ein gemeinsames Schema
- Harmonisiert Datentypen und Werte
- Markiert Herkunft klar (`source_period`, `source_dataset`, `source_type`)
- Markiert potenzielle Dubletten (loescht sie aber nicht)
- Haengt die Legacy-Faelle direkt an den Master an
- Speichert den aktualisierten Master wieder als `ec_antitrust_master.csv`


**Warum direkt in den Master?**
Die bestehende Masterdatei ist das zentrale Zielobjekt. Die pre-1997-Faelle werden nicht in eine
separate Datei geschrieben, sondern direkt integriert, damit `ec_antitrust_master.csv` die
vollstaendige EC-Basis bleibt.

**Warum keine Comfort Letters?**
Comfort Letters sind keine formalen Entscheidungen und werden in diesem Schritt bewusst ausgeschlossen.

## 1. Imports und Konfiguration

In [24]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
print("Imports OK")

Imports OK


## 2. Input- und Output-Pfade

Der aktualisierte Master wird wieder unter demselben Pfad gespeichert.
Optional wird ein Backup der alten Masterdatei angelegt.

In [25]:
MASTER_PATH = Path("data/processed/ec_antitrust_master.csv")
LEGACY_PATH = Path("data/processed/ec_legacy_formal_decisions.csv")
OUTPUT_DIR  = Path("data/processed")

OUTPUT_MASTER     = OUTPUT_DIR / "ec_antitrust_master.csv"
OUTPUT_BACKUP     = OUTPUT_DIR / "ec_antitrust_master_before_pre1997_merge.csv"
OUTPUT_DUPLICATES = OUTPUT_DIR / "ec_master_potential_duplicates.csv"

print(f"Master Path  : {MASTER_PATH}")
print(f"Legacy Path  : {LEGACY_PATH}")
print(f"Output Dir   : {OUTPUT_DIR}")

Master Path  : data\processed\ec_antitrust_master.csv
Legacy Path  : data\processed\ec_legacy_formal_decisions.csv
Output Dir   : data\processed


## 3. Daten laden

Beide CSVs werden geladen. Falls eine Datei fehlt, wird eine klare Fehlermeldung
ausgegeben und das Notebook stoppt.

In [26]:
def load_csv_strict(path: Path, label: str) -> pd.DataFrame:
    """Laedt eine CSV-Datei. Bricht mit klarer Fehlermeldung ab, falls die Datei fehlt."""
    if not path.exists():
        raise FileNotFoundError(
            f"[FEHLER] Datei nicht gefunden: {path}\n"
            f"Bitte stelle sicher, dass '{label}' erzeugt wurde, bevor dieses Notebook ausgefuehrt wird."
        )
    df = pd.read_csv(path, dtype=str)
    print(f"[OK] {label} geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
    return df

df_master = load_csv_strict(MASTER_PATH, "ec_antitrust_master.csv")
df_legacy = load_csv_strict(LEGACY_PATH, "ec_legacy_formal_decisions.csv")

[OK] ec_antitrust_master.csv geladen: 1185 Zeilen, 38 Spalten
[OK] ec_legacy_formal_decisions.csv geladen: 437 Zeilen, 21 Spalten


In [27]:
print("=== EC Antitrust Master - Spalten ===")
print(df_master.columns.tolist())
print()
print("=== Legacy Formal Decisions - Spalten ===")
print(df_legacy.columns.tolist())

=== EC Antitrust Master - Spalten ===
['ec_master_id', 'source_period', 'source_dataset', 'source_type', 'ec_case_id_raw', 'ec_case_id_normalized', 'case_number_raw', 'case_number_normalized', 'case_title', 'case_title_normalized', 'case_type', 'case_instrument', 'case_cartel', 'case_initiation_date', 'case_last_decision_date', 'decision_date_raw', 'decision_date_normalized', 'decision_year', 'publication_ref_raw', 'publication_ref_normalized', 'celex_raw', 'celex_normalized', 'language', 'case_dg', 'companies_raw', 'legal_basis_raw', 'sectors_raw', 'attachments_count', 'decisions_count', 'has_attachments', 'has_decisions', 'document_url', 'published_flag', 'official_journal_flag', 'source_url', 'source_file', 'is_potential_duplicate', 'duplicate_reason']

=== Legacy Formal Decisions - Spalten ===
['legacy_record_id', 'year_page', 'source_url', 'case_title', 'case_detail_text_raw', 'decision_date_raw', 'decision_type_raw', 'publication_ref_raw', 'case_number_raw', 'celex_raw', 'celex_d

In [28]:
print("=== EC Antitrust Master - Vorschau ===")
df_master.head(3)

=== EC Antitrust Master - Vorschau ===


,ec_master_id,source_period,source_dataset,source_type,ec_case_id_raw,ec_case_id_normalized,case_number_raw,case_number_normalized,case_title,case_title_normalized,case_type,case_instrument,case_cartel,case_initiation_date,case_last_decision_date,decision_date_raw,decision_date_normalized,decision_year,publication_ref_raw,publication_ref_normalized,celex_raw,celex_normalized,language,case_dg,companies_raw,legal_basis_raw,sectors_raw,attachments_count,decisions_count,has_attachments,has_decisions,document_url,published_flag,official_journal_flag,source_url,source_file,is_potential_duplicate,duplicate_reason
0,ECM-000001,modern,ec_antitrust_master,ec_modern,AT.35803,AT.35803,NaN,NaN,IPEX Consortium,ipex consortium,AtStandardATCCase,Antitrust & Cartels,Antitrust,1995-10-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en,Competition DG,Andrew Weir Shipping | DSR-Senator Lines | Compagnie Maritime d'affretement,Art. 101 TFEU,H.50.20 - Sea and coastal freight water transport,1,0,True,False,NaN,NaN,NaN,NaN,case-data-AT.json,False,NaN
1,ECM-000002,modern,ec_antitrust_master,ec_modern,AT.34950,AT.34950,NaN,NaN,ECO EMBALLAGES,eco emballages,AtStandardATCCase,Antitrust & Cartels,Antitrust,1993-12-17,2001-06-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en,Competition DG,NaN,Art. 101 TFEU,E.38.3 - Materials recovery,0,1,False,True,NaN,NaN,NaN,NaN,case-data-AT.json,False,NaN
2,ECM-000003,modern,ec_antitrust_master,ec_modern,AT.39172,AT.39172,NaN,NaN,Electricity sector inquiry,electricity sector inquiry,AtSectorInquiryCase,Antitrust & Cartels,Antitrust,2005-03-04,2007-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en,Competition DG,NaN,NaN,"D.35.1 - Electric power generation, transmission and distribution",0,1,False,True,NaN,NaN,NaN,NaN,case-data-AT.json,False,NaN


In [29]:
print("=== Legacy Formal Decisions - Vorschau ===")
df_legacy.head(3)

=== Legacy Formal Decisions - Vorschau ===


,legacy_record_id,year_page,source_url,case_title,case_detail_text_raw,decision_date_raw,decision_type_raw,publication_ref_raw,case_number_raw,celex_raw,celex_display,celex_complete,document_url,document_url_de,published_flag,official_journal_flag,legal_basis_raw,notes_raw,extraction_year,source_type,parse_success
0,1964_0001,1964,https://ec.europa.eu/competition/antitrust/closed/en/1964.html,Deca,22.10.1964 \n\nNegative clearance Art.81(1) [ex 85(1)] Official Journal :\...,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],L - 31/10/1964 Page : 2761,IV/71,364D0599,364D05\n99,31964D0599,https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0599,https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0599,True,True,Art.81,NaN,2024,formal_decision,True
1,1964_0002,1964,https://ec.europa.eu/competition/antitrust/closed/en/1964.html,Grundig-Consten,23.09.1964 \n\nInfringement Art.81 [ex 85] Official Journal :\n\nL - 20/10...,23.09.1964,Infringement Art.81 [ex 85],L - 20/10/1964 Page : 2545,IV/3344; IV/4,364D0566,364D05\n66,31964D0566,https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0566,https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0566,True,True,Art.81,NaN,2024,formal_decision,True
2,1964_0003,1964,https://ec.europa.eu/competition/antitrust/closed/en/1964.html,Nicholas Freres + Vitapro,30.07.1964 \n\nNegative clearance Art.81(1) [ex 85(1)] Official Journal :\...,30.07.1964,Negative clearance Art.81(1) [ex 85(1)],L - 26/08/1964 Page : 2287,IV/95,364D0502,364D05\n02,31964D0502,https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0502,https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0502,True,True,Art.81,NaN,2024,formal_decision,True


## 4. Spaltenvergleich

Hier wird transparent gezeigt, welche Spalten nur in einer der beiden Quellen
vorkommen und welche gemeinsam sind.

In [30]:
cols_master = set(df_master.columns)
cols_legacy = set(df_legacy.columns)

only_master = sorted(cols_master - cols_legacy)
only_legacy = sorted(cols_legacy - cols_master)
common      = sorted(cols_master & cols_legacy)

print(f"Nur in Master          ({len(only_master)}): {only_master}")
print()
print(f"Nur in Legacy          ({len(only_legacy)}): {only_legacy}")
print()
print(f"Gemeinsame Spalten     ({len(common)}): {common}")

Nur in Master          (27): ['attachments_count', 'case_cartel', 'case_dg', 'case_initiation_date', 'case_instrument', 'case_last_decision_date', 'case_number_normalized', 'case_title_normalized', 'case_type', 'celex_normalized', 'companies_raw', 'decision_date_normalized', 'decision_year', 'decisions_count', 'duplicate_reason', 'ec_case_id_normalized', 'ec_case_id_raw', 'ec_master_id', 'has_attachments', 'has_decisions', 'is_potential_duplicate', 'language', 'publication_ref_normalized', 'sectors_raw', 'source_dataset', 'source_file', 'source_period']

Nur in Legacy          (10): ['case_detail_text_raw', 'celex_complete', 'celex_display', 'decision_type_raw', 'document_url_de', 'extraction_year', 'legacy_record_id', 'notes_raw', 'parse_success', 'year_page']

Gemeinsame Spalten     (11): ['case_number_raw', 'case_title', 'celex_raw', 'decision_date_raw', 'document_url', 'legal_basis_raw', 'official_journal_flag', 'publication_ref_raw', 'published_flag', 'source_type', 'source_url']


## 5. Normalisierungsfunktionen

Einfache, defensive Hilfsfunktionen. Keine aggressiven inhaltlichen Aenderungen -
nur Whitespace-Bereinigung und Vereinheitlichung.

In [31]:
def clean_text(x) -> str:
    if pd.isna(x) or str(x).strip() == "":
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_title(x) -> str:
    t = clean_text(x)
    return t.lower() if t else ""

def normalize_case_number(x) -> str:
    t = clean_text(x)
    if not t:
        return ""
    t = re.sub(r"\s*/\s*", "/", t)
    return t.upper()

def normalize_ec_case_id(x) -> str:
    t = clean_text(x)
    if not t:
        return ""
    t = re.sub(r"\s*/\s*", "/", t)
    return t.upper()

def normalize_publication_ref(x) -> str:
    t = clean_text(x)
    return re.sub(r"\s+", " ", t) if t else ""

def normalize_celex(x) -> str:
    t = clean_text(x)
    return re.sub(r"\s+", "", t).upper() if t else ""

def normalize_date(x) -> str:
    t = clean_text(x)
    if not t:
        return ""
    m = re.search(r"(\d{4})[-./](\d{1,2})[-./](\d{1,2})", t)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    m2 = re.search(r"(\d{1,2})\.(\d{1,2})\.(\d{4})", t)
    if m2:
        return f"{m2.group(3)}-{int(m2.group(2)):02d}-{int(m2.group(1)):02d}"
    return t

def extract_year(date_str: str) -> str:
    m = re.match(r"(\d{4})", str(date_str))
    return m.group(1) if m else ""

print("Normalisierungsfunktionen definiert.")

Normalisierungsfunktionen definiert.


## 6. Gemeinsames Zielschema definieren

Alle Spalten des gemeinsamen Schemas werden hier definiert.
Fehlende Spalten werden mit leerem Wert ergaenzt.

In [32]:
TARGET_SCHEMA = [
    "ec_master_id",
    "source_period",
    "source_dataset",
    "source_type",
    "ec_case_id_raw",
    "ec_case_id_normalized",
    "case_number_raw",
    "case_number_normalized",
    "case_title",
    "case_title_normalized",
    "case_type",
    "case_instrument",
    "case_cartel",
    "case_initiation_date",
    "case_last_decision_date",
    "decision_date_raw",
    "decision_date_normalized",
    "decision_year",
    "publication_ref_raw",
    "publication_ref_normalized",
    "celex_raw",
    "celex_normalized",
    "language",
    "case_dg",
    "companies_raw",
    "legal_basis_raw",
    "sectors_raw",
    "attachments_count",
    "decisions_count",
    "has_attachments",
    "has_decisions",
    "document_url",
    "published_flag",
    "official_journal_flag",
    "source_url",
    "source_file",
    "is_potential_duplicate",
    "duplicate_reason",
]

print(f"Zielschema: {len(TARGET_SCHEMA)} Spalten")
print(TARGET_SCHEMA)

Zielschema: 38 Spalten
['ec_master_id', 'source_period', 'source_dataset', 'source_type', 'ec_case_id_raw', 'ec_case_id_normalized', 'case_number_raw', 'case_number_normalized', 'case_title', 'case_title_normalized', 'case_type', 'case_instrument', 'case_cartel', 'case_initiation_date', 'case_last_decision_date', 'decision_date_raw', 'decision_date_normalized', 'decision_year', 'publication_ref_raw', 'publication_ref_normalized', 'celex_raw', 'celex_normalized', 'language', 'case_dg', 'companies_raw', 'legal_basis_raw', 'sectors_raw', 'attachments_count', 'decisions_count', 'has_attachments', 'has_decisions', 'document_url', 'published_flag', 'official_journal_flag', 'source_url', 'source_file', 'is_potential_duplicate', 'duplicate_reason']


## 7. Master harmonisieren

Die bestehende Masterdatei wird auf das Zielschema gebracht. Fehlende Spalten werden ergaenzt.

In [33]:
def _s(df, col, default=""):
    """Hilfsfunktion: Spalte aus df holen oder leere Series zurueckgeben."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)

def harmonize_master(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["ec_master_id"]   = ""
    out["source_period"]  = _s(df, "source_period",  "modern").fillna("modern")
    out["source_dataset"] = _s(df, "source_dataset", "ec_antitrust_master").fillna("ec_antitrust_master")
    out["source_type"]    = _s(df, "source_type",    "ec_modern").fillna("ec_modern")

    ec_id_col = "ec_case_id" if "ec_case_id" in df.columns else "ec_case_id_raw"
    out["ec_case_id_raw"]         = _s(df, ec_id_col).apply(clean_text)
    out["ec_case_id_normalized"]  = out["ec_case_id_raw"].apply(normalize_ec_case_id)
    out["case_number_raw"]        = _s(df, "case_number_raw").apply(clean_text)
    out["case_number_normalized"] = out["case_number_raw"].apply(normalize_case_number)

    out["case_title"]            = _s(df, "case_title").apply(clean_text)
    out["case_title_normalized"] = out["case_title"].apply(normalize_title)

    out["case_type"]               = _s(df, "case_type").apply(clean_text)
    out["case_instrument"]         = _s(df, "case_instrument").apply(clean_text)
    out["case_cartel"]             = _s(df, "case_cartel").apply(clean_text)
    out["case_initiation_date"]    = _s(df, "case_initiation_date").apply(clean_text)
    out["case_last_decision_date"] = _s(df, "case_last_decision_date").apply(clean_text)

    out["decision_date_raw"]        = _s(df, "decision_date_raw").apply(clean_text)
    out["decision_date_normalized"] = out["decision_date_raw"].apply(normalize_date)
    out["decision_year"]            = out["decision_date_normalized"].apply(extract_year)

    out["publication_ref_raw"]        = _s(df, "publication_ref_raw").apply(clean_text)
    out["publication_ref_normalized"] = out["publication_ref_raw"].apply(normalize_publication_ref)
    out["celex_raw"]        = _s(df, "celex_raw").apply(clean_text)
    out["celex_normalized"] = out["celex_raw"].apply(normalize_celex)

    out["language"]          = _s(df, "language").apply(clean_text)
    out["case_dg"]           = _s(df, "case_dg").apply(clean_text)
    out["companies_raw"]     = _s(df, "companies_raw").apply(clean_text)
    out["legal_basis_raw"]   = _s(df, "legal_basis_raw").apply(clean_text)
    out["sectors_raw"]       = _s(df, "sectors_raw").apply(clean_text)
    out["attachments_count"] = _s(df, "attachments_count").fillna("")
    out["decisions_count"]   = _s(df, "decisions_count").fillna("")
    out["has_attachments"]   = _s(df, "has_attachments").fillna("")
    out["has_decisions"]     = _s(df, "has_decisions").fillna("")
    out["document_url"]      = _s(df, "document_url").fillna("")
    out["published_flag"]    = _s(df, "published_flag").fillna("")
    out["official_journal_flag"] = _s(df, "official_journal_flag").fillna("")
    out["source_url"]        = _s(df, "source_url").fillna("")
    out["source_file"]       = _s(df, "source_file", str(MASTER_PATH)).fillna(str(MASTER_PATH))

    out["is_potential_duplicate"] = False
    out["duplicate_reason"]       = ""

    return out[TARGET_SCHEMA].reset_index(drop=True)

df_master_h = harmonize_master(df_master)
print(f"Master harmonisiert: {df_master_h.shape}")
df_master_h.head(2)

Master harmonisiert: (1185, 38)


,ec_master_id,source_period,source_dataset,source_type,ec_case_id_raw,ec_case_id_normalized,case_number_raw,case_number_normalized,case_title,case_title_normalized,case_type,case_instrument,case_cartel,case_initiation_date,case_last_decision_date,decision_date_raw,decision_date_normalized,decision_year,publication_ref_raw,publication_ref_normalized,celex_raw,celex_normalized,language,case_dg,companies_raw,legal_basis_raw,sectors_raw,attachments_count,decisions_count,has_attachments,has_decisions,document_url,published_flag,official_journal_flag,source_url,source_file,is_potential_duplicate,duplicate_reason
0,,modern,ec_antitrust_master,ec_modern,AT.35803,AT.35803,,,IPEX Consortium,ipex consortium,AtStandardATCCase,Antitrust & Cartels,Antitrust,1995-10-20,,,,,,,,,en,Competition DG,Andrew Weir Shipping | DSR-Senator Lines | Compagnie Maritime d'affretement,Art. 101 TFEU,H.50.20 - Sea and coastal freight water transport,1,0,True,False,,,,,case-data-AT.json,False,
1,,modern,ec_antitrust_master,ec_modern,AT.34950,AT.34950,,,ECO EMBALLAGES,eco emballages,AtStandardATCCase,Antitrust & Cartels,Antitrust,1993-12-17,2001-06-15,,,,,,,,en,Competition DG,,Art. 101 TFEU,E.38.3 - Materials recovery,0,1,False,True,,,,,case-data-AT.json,False,


## 8. Legacy Formal Decisions harmonisieren

Die Legacy-Daten werden auf dasselbe Zielschema gebracht. Herkunft wird klar markiert.

In [34]:
def harmonize_legacy(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["ec_master_id"]   = ""
    out["source_period"]  = "pre1997"
    out["source_dataset"] = "ec_legacy_formal_decisions"
    out["source_type"]    = _s(df, "source_type", "ec_legacy_formal").fillna("ec_legacy_formal")

    out["ec_case_id_raw"]         = _s(df, "ec_case_id_raw").apply(clean_text)
    out["ec_case_id_normalized"]  = out["ec_case_id_raw"].apply(normalize_ec_case_id)
    out["case_number_raw"]        = _s(df, "case_number_raw").apply(clean_text)
    out["case_number_normalized"] = out["case_number_raw"].apply(normalize_case_number)

    out["case_title"]            = _s(df, "case_title").apply(clean_text)
    out["case_title_normalized"] = out["case_title"].apply(normalize_title)

    out["case_type"]               = ""
    out["case_instrument"]         = ""
    out["case_cartel"]             = ""
    out["case_initiation_date"]    = ""
    out["case_last_decision_date"] = ""

    out["decision_date_raw"]        = _s(df, "decision_date_raw").apply(clean_text)
    out["decision_date_normalized"] = out["decision_date_raw"].apply(normalize_date)
    out["decision_year"]            = out["decision_date_normalized"].apply(extract_year)

    out["publication_ref_raw"]        = _s(df, "publication_ref_raw").apply(clean_text)
    out["publication_ref_normalized"] = out["publication_ref_raw"].apply(normalize_publication_ref)
    out["celex_raw"]        = _s(df, "celex_raw").apply(clean_text)
    out["celex_normalized"] = out["celex_raw"].apply(normalize_celex)

    out["language"]          = ""
    out["case_dg"]           = ""
    out["companies_raw"]     = ""
    out["legal_basis_raw"]   = _s(df, "legal_basis_raw").apply(clean_text)
    out["sectors_raw"]       = ""
    out["attachments_count"] = ""
    out["decisions_count"]   = ""
    out["has_attachments"]   = ""
    out["has_decisions"]     = ""
    out["document_url"]      = _s(df, "document_url").fillna("")
    out["published_flag"]    = _s(df, "published_flag").fillna("")
    out["official_journal_flag"] = _s(df, "official_journal_flag").fillna("")
    out["source_url"]        = _s(df, "source_url").fillna("")
    out["source_file"]       = str(LEGACY_PATH)

    out["is_potential_duplicate"] = False
    out["duplicate_reason"]       = ""

    return out[TARGET_SCHEMA].reset_index(drop=True)

df_legacy_h = harmonize_legacy(df_legacy)
print(f"Legacy harmonisiert: {df_legacy_h.shape}")
df_legacy_h.head(2)

Legacy harmonisiert: (437, 38)


,ec_master_id,source_period,source_dataset,source_type,ec_case_id_raw,ec_case_id_normalized,case_number_raw,case_number_normalized,case_title,case_title_normalized,case_type,case_instrument,case_cartel,case_initiation_date,case_last_decision_date,decision_date_raw,decision_date_normalized,decision_year,publication_ref_raw,publication_ref_normalized,celex_raw,celex_normalized,language,case_dg,companies_raw,legal_basis_raw,sectors_raw,attachments_count,decisions_count,has_attachments,has_decisions,document_url,published_flag,official_journal_flag,source_url,source_file,is_potential_duplicate,duplicate_reason
0,,pre1997,ec_legacy_formal_decisions,formal_decision,,,IV/71,IV/71,Deca,deca,,,,,,22.10.1964,1964-10-22,1964,L - 31/10/1964 Page : 2761,L - 31/10/1964 Page : 2761,364D0599,364D0599,,,,Art.81,,,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0599,True,True,https://ec.europa.eu/competition/antitrust/closed/en/1964.html,data\processed\ec_legacy_formal_decisions.csv,False,
1,,pre1997,ec_legacy_formal_decisions,formal_decision,,,IV/3344; IV/4,IV/3344; IV/4,Grundig-Consten,grundig-consten,,,,,,23.09.1964,1964-09-23,1964,L - 20/10/1964 Page : 2545,L - 20/10/1964 Page : 2545,364D0566,364D0566,,,,Art.81,,,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0566,True,True,https://ec.europa.eu/competition/antitrust/closed/en/1964.html,data\processed\ec_legacy_formal_decisions.csv,False,


## 9. Technische Master-ID behandeln

Falls `ec_master_id` bereits in der Masterdatei existiert, wird sie beibehalten.
Fuer neue Legacy-Zeilen werden neue IDs fortlaufend ergaenzt.

In [35]:
def assign_master_ids(df_mh: pd.DataFrame, df_lh: pd.DataFrame):
    """Vergibt stabile ec_master_id fuer alle Zeilen."""
    existing_ids = pd.Series(dtype=str)
    if "ec_master_id" in df_master.columns:
        existing_ids = df_master["ec_master_id"].dropna()
        existing_ids = existing_ids[existing_ids.astype(str).str.strip().ne("")]

    if len(existing_ids) == len(df_mh):
        df_mh = df_mh.copy()
        df_mh["ec_master_id"] = existing_ids.values
        max_id = len(df_mh)
        print(f"Bestehende IDs beibehalten: {len(df_mh)} Eintraege")
    else:
        df_mh = df_mh.copy()
        df_mh["ec_master_id"] = [f"ECM-{i+1:06d}" for i in range(len(df_mh))]
        max_id = len(df_mh)
        print(f"Neue IDs fuer Master vergeben: {len(df_mh)} Eintraege")

    df_lh = df_lh.copy()
    df_lh["ec_master_id"] = [f"ECM-{max_id+i+1:06d}" for i in range(len(df_lh))]
    print(f"Neue IDs fuer Legacy vergeben: {len(df_lh)} Eintraege")
    return df_mh, df_lh

df_master_h, df_legacy_h = assign_master_ids(df_master_h, df_legacy_h)
print("Beispiel Master-IDs:")
print(df_master_h[["ec_master_id", "source_period", "case_title"]].head(3).to_string(index=False))
print("Beispiel Legacy-IDs:")
print(df_legacy_h[["ec_master_id", "source_period", "case_title"]].head(3).to_string(index=False))

Bestehende IDs beibehalten: 1185 Eintraege
Neue IDs fuer Legacy vergeben: 437 Eintraege
Beispiel Master-IDs:
ec_master_id source_period                 case_title
  ECM-000001        modern            IPEX Consortium
  ECM-000002        modern             ECO EMBALLAGES
  ECM-000003        modern Electricity sector inquiry
Beispiel Legacy-IDs:
ec_master_id source_period                case_title
  ECM-001186       pre1997                      Deca
  ECM-001187       pre1997           Grundig-Consten
  ECM-001188       pre1997 Nicholas Freres + Vitapro


## 10. Dublettenprüfung

Potenzielle Dubletten werden **markiert, aber nicht geloescht**.

Kriterien:
1. Gleiche `ec_case_id_normalized` (nicht leer)
2. Gleiche `case_number_normalized` (nicht leer)
3. Gleiche `celex_normalized` (nicht leer)
4. Gleiche `publication_ref_normalized` (nicht leer)
5. Gleicher `case_title_normalized` + gleiches `decision_year`

In [36]:
df_combined = pd.concat([df_master_h, df_legacy_h], ignore_index=True)
print(f"Kombiniert: {df_combined.shape[0]} Zeilen, {df_combined.shape[1]} Spalten")

Kombiniert: 1622 Zeilen, 38 Spalten


In [37]:
def mark_duplicates(df, col, reason_label):
    """Markiert Zeilen als potenzielle Dubletten, wenn col nicht leer und mehrfach vorkommt."""
    mask_nonempty = df[col].astype(str).str.strip().ne("")
    counts = df.loc[mask_nonempty, col].value_counts()
    dup_vals = counts[counts > 1].index
    dup_mask = df[col].isin(dup_vals) & mask_nonempty
    df.loc[dup_mask, "is_potential_duplicate"] = True
    df.loc[dup_mask, "duplicate_reason"] = df.loc[dup_mask, "duplicate_reason"].apply(
        lambda x: (str(x) + "; " + reason_label).lstrip("; ") if x else reason_label
    )
    return df

df_combined = mark_duplicates(df_combined, "ec_case_id_normalized",      "same_ec_case_id")
df_combined = mark_duplicates(df_combined, "case_number_normalized",     "same_case_number")
df_combined = mark_duplicates(df_combined, "celex_normalized",           "same_celex")
df_combined = mark_duplicates(df_combined, "publication_ref_normalized", "same_pub_ref")

# Titel + Jahr
title_year_key = (
    df_combined["case_title_normalized"].astype(str).str.strip() + "||" +
    df_combined["decision_year"].astype(str).str.strip()
)
mask_valid = (
    df_combined["case_title_normalized"].astype(str).str.strip().ne("") &
    df_combined["decision_year"].astype(str).str.strip().ne("")
)
counts_ty = title_year_key[mask_valid].value_counts()
dup_ty    = counts_ty[counts_ty > 1].index
dup_mask_ty = title_year_key.isin(dup_ty) & mask_valid
df_combined.loc[dup_mask_ty, "is_potential_duplicate"] = True
df_combined.loc[dup_mask_ty, "duplicate_reason"] = df_combined.loc[dup_mask_ty, "duplicate_reason"].apply(
    lambda x: (str(x) + "; same_title_year").lstrip("; ") if x else "same_title_year"
)

n_dup = df_combined["is_potential_duplicate"].sum()
print(f"Potenzielle Dubletten markiert: {n_dup}")

Potenzielle Dubletten markiert: 874


In [38]:
df_duplicates = df_combined[df_combined["is_potential_duplicate"] == True].copy()
if len(df_duplicates) > 0:
    print(f"Potenzielle Dubletten ({len(df_duplicates)} Zeilen):")
    display(df_duplicates[["ec_master_id", "source_period", "source_dataset",
                            "case_title", "case_number_normalized",
                            "celex_normalized", "decision_year",
                            "duplicate_reason"]].head(20))
else:
    print("Keine potenziellen Dubletten gefunden.")

Potenzielle Dubletten (874 Zeilen):


,ec_master_id,source_period,source_dataset,case_title,case_number_normalized,celex_normalized,decision_year,duplicate_reason
748,ECM-000749,pre1997,ec_legacy_formal_decisions,Deca,IV/71,364D0599,1964,same_case_number; same_celex; same_pub_ref; same_title_year
749,ECM-000750,pre1997,ec_legacy_formal_decisions,Grundig-Consten,IV/3344; IV/4,364D0566,1964,same_case_number; same_celex; same_pub_ref; same_title_year
750,ECM-000751,pre1997,ec_legacy_formal_decisions,Nicholas Freres + Vitapro,IV/95,364D0502,1964,same_case_number; same_celex; same_pub_ref; same_title_year
751,ECM-000752,pre1997,ec_legacy_formal_decisions,Bendix + Mertens and Straat,IV/12868,364D0344,1964,same_case_number; same_celex; same_pub_ref; same_title_year
752,ECM-000753,pre1997,ec_legacy_formal_decisions,Grosfillex + Fillistorf,IV/61,364D0233,1964,same_case_number; same_celex; same_pub_ref; same_title_year
753,ECM-000754,pre1997,ec_legacy_formal_decisions,Maison Jalatte,IV/22491,366D0005,1965,same_case_number; same_celex; same_pub_ref; same_title_year
754,ECM-000755,pre1997,ec_legacy_formal_decisions,Hummel - Isbecque,IV/2702,365D0426,1965,same_case_number; same_celex; same_pub_ref; same_title_year
755,ECM-000756,pre1997,ec_legacy_formal_decisions,Dru-Blondel,IV/3036,365D0366,1965,same_case_number; same_celex; same_pub_ref; same_title_year
756,ECM-000757,pre1997,ec_legacy_formal_decisions,Transocean Marine Paint Association,IV/223,367D0454,1967,same_case_number; same_celex; same_pub_ref; same_title_year
757,ECM-000758,pre1997,ec_legacy_formal_decisions,CFA,IV/666,368D0377,1968,same_case_number; same_celex; same_pub_ref; same_title_year


## 11. Direkte Integration in den Master

Der kombinierte DataFrame wird als aktualisierter Master betrachtet.

In [39]:
df_ec_master_updated = df_combined.copy()

print(f"Aktualisierter Master: {df_ec_master_updated.shape[0]} Zeilen, {df_ec_master_updated.shape[1]} Spalten")
print(f"  davon modern (original): {(df_ec_master_updated['source_period'] != 'pre1997').sum()}")
print(f"  davon pre1997 (neu):     {(df_ec_master_updated['source_period'] == 'pre1997').sum()}")

Aktualisierter Master: 1622 Zeilen, 38 Spalten
  davon modern (original): 748
  davon pre1997 (neu):     874


## 12. Qualitaetschecks

In [40]:
n_original = len(df_master)
n_legacy   = len(df_legacy)
n_updated  = len(df_ec_master_updated)
n_new      = (df_ec_master_updated["source_period"] == "pre1997").sum()
n_celex    = df_ec_master_updated["celex_normalized"].astype(str).str.strip().ne("").sum()
n_pub_ref  = df_ec_master_updated["publication_ref_normalized"].astype(str).str.strip().ne("").sum()
n_doc_url  = df_ec_master_updated["document_url"].astype(str).str.strip().ne("").sum()
n_dup      = df_ec_master_updated["is_potential_duplicate"].sum()

print("=== Qualitaetschecks ===")
print(f"Faelle in urspruenglichem Master    : {n_original}")
print(f"Faelle in Legacy Formal Decisions   : {n_legacy}")
print(f"Faelle im aktualisierten Master     : {n_updated}")
print(f"Neu hinzugefuegte pre-1997-Faelle   : {n_new}")
print(f"Faelle mit CELEX                    : {n_celex}")
print(f"Faelle mit Publikationsreferenz     : {n_pub_ref}")
print(f"Faelle mit Dokumentlink             : {n_doc_url}")
print(f"Potenzielle Dubletten               : {n_dup}")

=== Qualitaetschecks ===
Faelle in urspruenglichem Master    : 1185
Faelle in Legacy Formal Decisions   : 437
Faelle im aktualisierten Master     : 1622
Neu hinzugefuegte pre-1997-Faelle   : 874
Faelle mit CELEX                    : 766
Faelle mit Publikationsreferenz     : 766
Faelle mit Dokumentlink             : 766
Potenzielle Dubletten               : 874


In [41]:
print("=== Fallzahlen pro Jahr ===")
year_counts = df_ec_master_updated["decision_year"].value_counts().sort_index()
print(year_counts.to_string())

=== Fallzahlen pro Jahr ===
decision_year
        748
1964     10
1965      6
1967      2
1968     16
1969     20
1970     14
1971     36
1972     30
1973     16
1974     22
1975     32
1976     16
1977     38
1978     24
1979     24
1980     18
1981     30
1982     26
1983     34
1984     40
1985     34
1986     42
1987     30
1988     48
1989     28
1990     32
1991     32
1992     64
1995     24
1996     44
1997     42


In [42]:
print("=== Fallzahlen nach source_period ===")
print(df_ec_master_updated["source_period"].value_counts().to_string())
print()
print("=== Fallzahlen nach source_dataset ===")
print(df_ec_master_updated["source_dataset"].value_counts().to_string())

=== Fallzahlen nach source_period ===
source_period
pre1997    874
modern     748

=== Fallzahlen nach source_dataset ===
source_dataset
ec_legacy_formal_decisions    874
ec_antitrust_master           748


## 13. Export

In [43]:
import shutil

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Backup der alten Masterdatei
if MASTER_PATH.exists():
    shutil.copy2(MASTER_PATH, OUTPUT_BACKUP)
    print(f"[OK] Backup erstellt: {OUTPUT_BACKUP}")

# Aktualisierter Master (gleicher Pfad)
df_ec_master_updated.to_csv(OUTPUT_MASTER, index=False)
print(f"[OK] Aktualisierter Master exportiert: {OUTPUT_MASTER}  ({len(df_ec_master_updated)} Zeilen)")

# Potenzielle Dubletten (optional)
if len(df_duplicates) > 0:
    df_duplicates.to_csv(OUTPUT_DUPLICATES, index=False)
    print(f"[OK] Dubletten exportiert: {OUTPUT_DUPLICATES}  ({len(df_duplicates)} Zeilen)")
else:
    print("[INFO] Keine Dubletten-Datei erstellt (keine potenziellen Dubletten gefunden).")

[OK] Backup erstellt: data\processed\ec_antitrust_master_before_pre1997_merge.csv
[OK] Aktualisierter Master exportiert: data\processed\ec_antitrust_master.csv  (1622 Zeilen)
[OK] Dubletten exportiert: data\processed\ec_master_potential_duplicates.csv  (874 Zeilen)


## 14. Abschluss / Naechste Schritte

### Was wurde erstellt?

- Die bestehende **`ec_antitrust_master.csv`** wurde um **pre-1997 Legacy Formal Decisions** erweitert.
- **Comfort Letters** wurden bewusst nicht beruecksichtigt.
- Die Herkunft jedes Eintrags ist klar markiert ueber:
  - `source_period` (`modern` / `pre1997`)
  - `source_dataset` (`ec_antitrust_master` / `ec_legacy_formal_decisions`)
  - `source_type`
- Potenzielle Dubletten sind markiert (Spalten `is_potential_duplicate`, `duplicate_reason`), aber **nicht geloescht**.
- Jeder Datensatz hat eine stabile technische ID (`ec_master_id`).

### Erzeugte Output-Dateien

- `data/processed/ec_antitrust_master.csv` – aktualisierter Master (inkl. pre-1997-Faelle)
- `data/processed/ec_master_potential_duplicates.csv` – nur Zeilen mit `is_potential_duplicate == True`
- `data/processed/ec_antitrust_master_before_pre1997_merge.csv` – Backup der alten Masterdatei (optional)

### Naechste Schritte

1. **CJEU-Seite aufbauen**: Auf Basis der aktualisierten `ec_antitrust_master.csv` kann nun die CJEU-Seite aufgebaut werden.
2. **CJEU-EC-Referenzen pruefen**: Danach kann geprueft werden, welche CJEU-Faelle auf EC-Faelle referenzieren.
3. **Dublettenbereinigung**: Die markierten potenziellen Dubletten koennen manuell geprueft und ggf. bereinigt werden.
4. **Comfort Letters**: Falls benoetigt, koennen Comfort Letters in einem separaten Schritt beruecksichtigt werden.